In [4]:
import pandas as pd
import numpy as np

from collections import Counter
from pprint import pprint
import regex as re


In [ ]:
df = pd.read_excel('./RCT_Results_Cleaned.xlsx')
df.head()

# prelim MC eda

In [ ]:
for col in df.columns:
    print(f"\n{'=' * 80}")
    print(f"Column: {col}")
    print(f"{'=' * 80}")

    counts = Counter(df[col].dropna())

    for value, count in counts.most_common():
        print(f"{count:>5} | {repr(value)}")

In [8]:
def pretty_group_counts(df, group_col, target_cols=None):
    """
    Pretty print value counts for columns grouped by group_col.
    Parameters
    ----------
    df : pd.DataFrame
    group_col : str
        Column to group by

    target_cols : str | list[str] | None
        Columns to analyze.
        - None -> all columns except group_col
        - str  -> single column
        - list -> multiple columns
    """

    # Default = all columns except grouping column
    if target_cols is None:
        target_cols = [c for c in df.columns if c != group_col]

    # Allow passing a single string column
    elif isinstance(target_cols, str):
        target_cols = [target_cols]

    for target_col in target_cols:

        if target_col not in df.columns:
            print(f"\n[WARNING] Column not found: {target_col}")
            continue

        print(f"\n\n{'#' * 100}")
        print(f"TARGET COLUMN: {target_col}")
        print(f"{'#' * 100}")

        grouped = (
            df.groupby(group_col)[target_col]
            .value_counts(dropna=False)
        )

        for group_value in grouped.index.get_level_values(0).unique():

            print(f"\n{'=' * 80}")
            print(f"{group_col}: {repr(group_value)}")
            print(f"{'=' * 80}")

            sub = grouped[group_value]

            # value_counts already sorted descending
            for response, count in sub.items():
                print(f"{count:>5} | {repr(response)}")



In [ ]:
pretty_group_counts(
    df,
    group_col="Condition")

# main columns of interests

In [10]:
col = 'Collaboration pattern observed *If multiple choices apply, use the “Other” freeform text field.'

df.loc[
    df[col].str.strip() == "2 & 3",
    col
] = (
    "2. Agent asked for human input less than 5 times; "
    "3. Human had to provide a minor suggestion or two to redirect agent on the right path"
)

### collaboration patterns
* the "not reproducing the target" isn't a "success metric" for our paper

In [ ]:
pretty_group_counts(
    df,
    group_col=["Condition"],
    target_cols=  'Collaboration pattern observed *If multiple choices apply, use the “Other” freeform text field.'
    )

In [ ]:
pretty_group_counts(
    df,
    group_col=["Condition", 'Results: Match'],
    target_cols=  'Collaboration pattern observed *If multiple choices apply, use the “Other” freeform text field.'
    )

In [ ]:
pretty_group_counts(
    df,
    group_col=["Condition", 'Results: Match', 'Reproduction failure mode classification (in case reproduction of the given target failed)'],
    target_cols=  'Collaboration pattern observed *If multiple choices apply, use the “Other” freeform text field.'
    )

### where agent added value


In [ ]:
pretty_group_counts(
    df,
    group_col="Condition",
    target_cols= 'Where Agent added value'
    )

In [ ]:
col = 'Where Agent added value'

bad = (
    "11. Interpreting intermediate results intelligently so that it could move on quickly to next steps, "
    "created wrapper (fixed the errors after built) under logs to record outputs, keeps main logic unchanged"
)

good = (
    "11. Interpreting intermediate results intelligently so that it could move on quickly to next steps"
)

df[col] = df[col].str.replace(bad, good, regex=False)

In [ ]:
col = 'Where Agent added value'

# Drop missing values
values = df[col].dropna()

all_items = []

for entry in values:
    # Split on numbered list items
    parts = re.split(r',\s*(?=\d+\.\s)', entry)

    # Clean whitespace
    parts = [p.strip() for p in parts if p.strip()]

    all_items.extend(parts)

# Aggregate counts
counts = Counter(all_items)

# Convert to dataframe
counts_df = (
    pd.DataFrame(counts.items(), columns=['Category', 'Count'])
    .sort_values('Count', ascending=False)
    .reset_index(drop=True)
)

(counts_df)

###

In [ ]:
pretty_group_counts(
    df,
    group_col=["Condition", 'Total duration (in minutes) Measure by: Start from difference between first and last timestamp (as provided by script), manually subtract lunch breaks etc. (afk time), add any additional time for analysis etc. after the last timestamp'],
    target_cols= ['Results: Match']
    )

In [ ]:
pretty_group_counts(
    df,
    group_col="Condition",
    target_cols= 'Results: Match'
    )

### Where agent struggled and needed help

In [ ]:
pretty_group_counts(
    df,
    group_col="Condition",
    target_cols= 'Where Agent struggled and needed help'
    )

In [ ]:
col = 'Where Agent struggled and needed help'

# Drop missing values
values = df[col].dropna()

all_items = []

for entry in values:
    # Split on numbered list items
    parts = re.split(r',\s*(?=\d+\.\s)', entry)

    # Clean whitespace
    parts = [p.strip() for p in parts if p.strip()]

    all_items.extend(parts)

# Aggregate counts
counts = Counter(all_items)

# Convert to dataframe
counts_df = (
    pd.DataFrame(counts.items(), columns=['Category', 'Count'])
    .sort_values('Count', ascending=False)
    .reset_index(drop=True)
)

(counts_df)



### Reproduction failure mode classification

In [ ]:
pretty_group_counts(
    df,
    group_col="Condition",
    target_cols=  'Reproduction failure mode classification (in case reproduction of the given target failed)'
    )

### other observations

In [ ]:
pretty_group_counts(
    df,
    group_col="Condition",
    target_cols= 'Other Notes (error messages, surprises, observations - anything that doesn\'t fit above)'
    )